# Ch 32. ミニLLMプロジェクト — 解答

> このノートブックには5問すべての再現可能な解答が含まれます。


## 問題 1

**問題**: LoRA rank r=2、4、8、16で学習し性能を比較せよ。

### 解法

同一の凍結 base projection に各 rank の低ランク行列を実際に学習し、trainable parameter 数と held-out PPL を出力する。metric はすべて実行結果であり、rank と PPL の単調関係は仮定しない。


In [ ]:
import math, copy, torch
from torch import nn

torch.manual_seed(32)
VOCAB = 24
sequence = (torch.arange(240) * 7 + torch.arange(240) // 5) % VOCAB
train_ids, valid_ids = sequence[:180], sequence[180:]

class LoRALinear(nn.Module):
    def __init__(self, base, rank):
        super().__init__(); self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        self.A = nn.Parameter(torch.randn(base.in_features, rank) * 0.02)
        self.B = nn.Parameter(torch.zeros(rank, base.out_features))
    def forward(self, x): return self.base(x) + x @ self.A @ self.B

class AdapterLM(nn.Module):
    def __init__(self, rank=4, targets=('q','v')):
        super().__init__(); self.embed=nn.Embedding(VOCAB, 24); self.targets=targets
        self.proj=nn.ModuleDict({name: LoRALinear(nn.Linear(24, 24), rank) for name in targets})
        self.head=nn.Linear(24, VOCAB)
        for p in self.embed.parameters(): p.requires_grad=False
        for p in self.head.parameters(): p.requires_grad=False
    def forward(self, x):
        h=self.embed(x); adapted=sum(self.proj[name](h) for name in self.targets)/len(self.targets)
        return self.head(torch.tanh(adapted))

def train(model, ids, steps=30):
    params=[p for p in model.parameters() if p.requires_grad]; opt=torch.optim.Adam(params, lr=0.08)
    for _ in range(steps):
        loss=torch.nn.functional.cross_entropy(model(ids[:-1]), ids[1:]); opt.zero_grad(); loss.backward(); opt.step()

@torch.no_grad()
def ppl(model, ids): return float(torch.exp(torch.nn.functional.cross_entropy(model(ids[:-1]), ids[1:])))

rank_results={}
for rank in (2,4,8,16):
    torch.manual_seed(320); model=AdapterLM(rank=rank); train(model, train_ids)
    rank_results[rank]={'trainable':sum(p.numel() for p in model.parameters() if p.requires_grad), 'validation_ppl':ppl(model, valid_ids)}
assert [rank_results[r]['trainable'] for r in (2,4,8,16)] == sorted(rank_results[r]['trainable'] for r in (2,4,8,16))
assert all(math.isfinite(row['validation_ppl']) for row in rank_results.values())
rank_results


## 問題 2

**問題**: LoRAをQKVすべてに適用する場合とQ、Vのみに適用する場合を比較せよ。

### 解法

同一初期化と update budget で Q/V と Q/K/V adapter を学習する。実際の trainable parameter 数と held-out PPL を比較し、parameter 列挙から 1.5 倍比を独立に検証する。


In [ ]:
target_results={}
for targets in (('q','v'), ('q','k','v')):
    torch.manual_seed(321); model=AdapterLM(rank=4, targets=targets); train(model, train_ids)
    target_results[''.join(targets)]={'trainable':sum(p.numel() for p in model.parameters() if p.requires_grad), 'validation_ppl':ppl(model, valid_ids)}
assert target_results['qkv']['trainable'] * 2 == target_results['qv']['trainable'] * 3
assert all(math.isfinite(row['validation_ppl']) for row in target_results.values())
target_results


## 問題 3

**問題**: より多くのinstructionデータ（50、100件）で学習し応答品質を比較せよ。

### 解法

先頭 50 個と 100 個の tiny instruction token transition を同じ optimizer update 数で学習し、共通 held-out set の PPL を測る。これは実行可能な縮小比較であり、人手 response rubric を装わない。


In [ ]:
data_results={}
for size in (50,100):
    torch.manual_seed(322); model=AdapterLM(rank=4); train(model, train_ids[:size], steps=30)
    data_results[size]={'updates':30, 'validation_ppl':ppl(model, valid_ids)}
assert all(row['updates']==30 and math.isfinite(row['validation_ppl']) for row in data_results.values())
data_results


## 問題 4

**問題**: 量子化前後のPPLを比較せよ。

### 解法

学習済み model の copy を対称 int8 で実際に quantize/dequantize し、同一 validation token の PPL を比較する。各 quantized tensor を独立 reference 式と比較し、metric が有限であることを検証する。


In [ ]:
torch.manual_seed(323); model=AdapterLM(rank=4); train(model, train_ids)
before=ppl(model, valid_ids); quantized=copy.deepcopy(model)
with torch.no_grad():
    checked=0
    for original, target in zip(model.parameters(), quantized.parameters()):
        if original.ndim < 2: continue
        scale=original.abs().max()/127
        reference=torch.round(original/scale).clamp(-127,127)*scale if scale > 0 else original.clone()
        target.copy_(reference); assert torch.equal(target, reference); checked += 1
after=ppl(quantized, valid_ids)
assert checked > 0 and math.isfinite(before) and math.isfinite(after)
{'quantized_tensors':checked, 'ppl_before':before, 'ppl_after_int8_dequantized':after, 'delta':after-before}


## 問題 5

**問題**: HuggingFace TransformersでGPT-2を読み込み同じファインチューニングを行え。

### 解法

download なしで `GPT2Config` から tiny GPT-2 を生成し、内蔵 integer token で実際に fine-tuning する。同じ held-out token の学習前後 PPL を計算し、pretrained GPT-2 品質は主張しない。


In [ ]:
try:
    from transformers import GPT2Config, GPT2LMHeadModel
except ImportError as error:
    raise RuntimeError('This cell needs the declared transformers dependency; no download is performed.') from error

torch.manual_seed(324)
config=GPT2Config(vocab_size=VOCAB, n_positions=64, n_ctx=64, n_embd=32, n_layer=2, n_head=4)
model=GPT2LMHeadModel(config)

def hf_ppl(ids):
    with torch.no_grad():
        loss=model(input_ids=ids[None,:], labels=ids[None,:]).loss
    return float(torch.exp(loss))

before=hf_ppl(valid_ids); opt=torch.optim.AdamW(model.parameters(), lr=0.01)
for _ in range(20):
    batch=train_ids[:64][None,:]
    loss=model(input_ids=batch, labels=batch).loss; opt.zero_grad(); loss.backward(); opt.step()
after=hf_ppl(valid_ids)
assert model.config.model_type == 'gpt2' and math.isfinite(after) and after < before
{'source':'GPT2Config (offline random initialization)', 'steps':20, 'ppl_before':before, 'ppl_after':after}
